In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

from pathlib import Path
WORKDIR = str(Path.cwd().parent) + '/'  # raiz do repositório
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/logs/"+data_hora

In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [5]:
df_csv_phone = pd.read_csv("activity_recognition_data_phone_join_multi_index.csv")

In [7]:
df_csv_phone.head()

,pessoa_x,atividade,tempo,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,252207918580802,-4.332779,13.361191,-0.718872,-0.853210,0.297226,0.890182
1,1600,0,252207968934806,-0.319443,13.318359,-0.232025,-0.875137,0.015472,0.162231
2,1600,0,252208019288809,1.566452,9.515274,-0.017776,-0.720169,0.388489,-0.284012
3,1600,0,252208069642813,-0.323746,5.262665,0.322342,-0.571640,1.227402,-0.241669
4,1600,0,252208119996817,-1.811676,3.710510,1.373932,-0.380493,1.202835,-0.213135


In [8]:
df_csv_phone.shape

(2787819, 9)

In [6]:
df_csv_watch = pd.read_csv("activity_recognition_data_watch_join_multi_index.csv")

In [9]:
df_csv_watch.shape

(3238804, 9)

In [10]:
df_csv_phone.columns

Index(['pessoa_x', 'atividade', 'tempo', 'acelerômetro_x', 'acelerômetro_Y',
       'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'],
      dtype='object')

In [ ]:
df_csv_phone.columns = ['pessoa_phone_gyro','atividade_gyro','tempo_gyro','giroscópio_phone_x','giroscópio_phone_Y','giroscópio_Z']

In [ ]:
df_csv_gyro.head()

In [ ]:
df_csv_gyro.shape

In [ ]:
df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [ ]:
df_csv_gyro.head()

In [ ]:
df_csv_gyro.info()

In [ ]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [ ]:
df_csv_accel.shape

In [ ]:
df_csv_accel.columns = ['pessoa_acelelômetro','atividade_acelerômetro','tempo_acelelômetro','acelelômetro_x','acelelômetro_Y','acelelômetro_Z']

In [ ]:
df_csv_accel.head()

In [ ]:
df_csv_accel.set_index(["pessoa_acelelômetro","atividade_acelerômetro","tempo_acelelômetro"],inplace=True)

In [ ]:
df_csv_accel.head()

In [ ]:
# Select rows with matching indices (inner join)
common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1)

In [ ]:
result.shape

In [ ]:
result.tail()

In [ ]:
result.reset_index(inplace=True)

In [ ]:
result.tail()

In [ ]:
result.columns = ['pessoa_x','atividade','tempo','acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'] 

In [ ]:
result.head()

In [ ]:
result.atividade.value_counts().sort_index()

In [ ]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [ ]:
result["atividade"] = result["atividade"].replace(to_replace=mapping_dict)

In [ ]:
result.head()

In [ ]:
result.tail()

In [ ]:
cols_to_check = ['acelerômetro_Z','giroscópio_Z']

In [ ]:
result[cols_to_check] = result[cols_to_check].replace({';':''}, regex=True)

In [ ]:
result.head()

In [ ]:
result.to_csv(WORKDIR + 'Data/processed/activity_recognition_data_watch_join_multi_index.csv', index=False)